# 01 — Data loading and validation

**Sweden vs England, UEFA Women's EURO 2025 quarter-final (17 July 2025, Zürich)**
2-2 after extra time, England won 3-2 on penalties.

This notebook:
1. finds the match in StatsBomb Open Data (competition 53, season 315),
2. loads events, lineups and 360 freeze frames (cached to `data/raw/` as parquet),
3. validates the data against known match facts and prints `OK` / `MISMATCH` for each check.

If the data disagrees with a known fact, the mismatch is reported as is — the code is not adjusted to force a match.

**Minute convention.** StatsBomb `minute`/`second` are the elapsed match clock. A goal at 1:44 is the
2nd minute on a broadcast (2'). All checks against "known facts" use `display_minute = minute + 1`,
and the raw clock is printed next to every check.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import data as d

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Find the match and load the data

In [2]:
matches = d.load_matches()
match_id = d.find_match_id(d.SWEDEN, d.ENGLAND)
match = matches[matches.match_id == match_id].iloc[0]
print(f"match_id: {match_id}")
match[["match_date", "kick_off", "competition_stage", "home_team", "away_team",
       "home_score", "away_score", "stadium", "referee", "match_status_360"]]

match_id: 4018355


match_date                    2025-07-17
kick_off                    19:00:00.000
competition_stage         Quarter-finals
home_team                 Sweden Women's
away_team                England Women's
home_score                             2
away_score                             2
stadium               Stadion Letzigrund
referee              Marta Huerta de Aza
match_status_360               available
Name: 9, dtype: object

In [3]:
events = d.load_events(match_id)
lineups = d.load_lineups(match_id)
frames = d.load_frames_360(match_id)

print(f"events:  {events.shape}")
print(f"lineups: {lineups.shape}")
print(f"360:     {frames.shape} rows, {frames.id.nunique()} events with a freeze frame "
      f"({frames.id.nunique() / len(events):.0%} of all events)")

events:  (3888, 102)
lineups: (46, 8)
360:     (50943, 7) rows, 3341 events with a freeze frame (86% of all events)


In [4]:
events.type.value_counts()

type
Pass                 1083
Ball Receipt*         864
Carry                 737
Pressure              418
Ball Recovery         157
Duel                  113
Block                  82
Goal Keeper            64
Clearance              61
Shot                   45
Dribble                36
Foul Committed         35
Foul Won               34
Miscontrol             29
Dispossessed           28
Dribbled Past          21
Interception           15
Substitution           12
Half Start             10
Half End               10
Injury Stoppage         6
Tactical Shift          5
50/50                   4
Player Off              4
Player On               4
Error                   3
Shield                  3
Starting XI             2
Referee Ball-Drop       2
Offside                 1
Name: count, dtype: int64

In [5]:
events.groupby("period").agg(
    first_clock=("timestamp", "min"), last_clock=("timestamp", "max"),
    min_minute=("minute", "min"), max_minute=("minute", "max"), n_events=("id", "size"),
)

,first_clock,last_clock,min_minute,max_minute,n_events
period,,,,,
1,00:00:00.000,00:46:37.813,0,46,1597
2,00:00:00.000,00:48:36.140,45,93,1450
3,00:00:00.000,00:18:31.983,90,108,423
4,00:00:00.000,00:18:07.767,105,123,386
5,00:00:00.000,00:10:12.979,120,130,32


## 2. Validation against known facts

Each check prints `OK` or `MISMATCH`; all results are collected in a summary table at the end.

In [6]:
results: list[dict] = []


def check(name: str, expected, actual, detail: str = "") -> None:
    """Compare an expected fact with the value found in the data and log the result."""
    status = "OK" if expected == actual else "MISMATCH"
    results.append(dict(check=name, expected=expected, actual=actual,
                        status=status, detail=detail))
    print(f"[{status:8}] {name}: expected={expected!r} actual={actual!r} {detail}")


def clock_str(row) -> str:
    return f"(clock {int(row.minute):02d}:{int(row.second):02d}, period {int(row.period)})"

### 2.1 Goals in play (periods 1–4)

Goals are shots with `shot_outcome == "Goal"` plus any `Own Goal For` events (there are none in this match,
but the check would include them).

In [7]:
in_play = events[events.period < d.SHOOTOUT_PERIOD]
goals = in_play[(in_play.type == "Shot") & (in_play.shot_outcome == "Goal")]
own_goals = in_play[in_play.type == "Own Goal For"]
goals = pd.concat([goals, own_goals]).sort_values("index")
goals.assign(display_minute=goals.minute + 1)[
    ["period", "minute", "second", "display_minute", "team", "player", "type", "shot_statsbomb_xg"]
]

,period,minute,second,display_minute,team,player,type,shot_statsbomb_xg
80,1,1,44,2,Sweden Women's,Kosovare Asllani,Shot,0.262062
858,1,24,39,25,Sweden Women's,Emma Stina Blackstenius,Shot,0.286011
2680,2,78,31,79,England Women's,Lucy Bronze,Shot,0.357736
2751,2,80,13,81,England Women's,Michelle Agyemang,Shot,0.499702


In [8]:
expected_goals = [
    (d.SWEDEN, 2, "Asllani"),
    (d.SWEDEN, 25, "Blackstenius"),
    (d.ENGLAND, 79, "Bronze"),
    (d.ENGLAND, 81, "Agyemang"),
]
for team, minute, surname in expected_goals:
    hit = goals[(goals.team == team) & goals.player.str.contains(surname, na=False)]
    if hit.empty:
        check(f"{surname} goal ({team})", f"{minute}'", None, "no goal by this player")
        continue
    row = hit.iloc[0]
    check(f"{surname} goal ({team})", f"{minute}'", f"{row.minute + 1}'", clock_str(row))

check("Number of goals in periods 1-4", 4, len(goals))

[OK      ] Asllani goal (Sweden Women's): expected="2'" actual="2'" (clock 01:44, period 1)
[OK      ] Blackstenius goal (Sweden Women's): expected="25'" actual="25'" (clock 24:39, period 1)
[OK      ] Bronze goal (England Women's): expected="79'" actual="79'" (clock 78:31, period 2)
[OK      ] Agyemang goal (England Women's): expected="81'" actual="81'" (clock 80:13, period 2)
[OK      ] Number of goals in periods 1-4: expected=4 actual=4 


### 2.2 Score after extra time

In [9]:
score = goals.team.value_counts()
check("Sweden goals after extra time", 2, int(score.get(d.SWEDEN, 0)))
check("England goals after extra time", 2, int(score.get(d.ENGLAND, 0)))
check("Match sheet score (home-away)", "2-2", f"{match.home_score}-{match.away_score}")
check("Extra time was played (periods 3 and 4 present)", True,
      bool({3, 4}.issubset(set(events.period))))

[OK      ] Sweden goals after extra time: expected=2 actual=2 
[OK      ] England goals after extra time: expected=2 actual=2 
[OK      ] Match sheet score (home-away): expected='2-2' actual='2-2' 
[OK      ] Extra time was played (periods 3 and 4 present): expected=True actual=True 


### 2.3 Penalty shootout (period 5)

A kick is counted as scored only when `shot_outcome == "Goal"`.

In [10]:
shootout = events[(events.period == d.SHOOTOUT_PERIOD) & (events.type == "Shot")]
shootout_goals = shootout[shootout.shot_outcome == "Goal"]

check("Shootout kicks", 14, len(shootout))
check("Shootout kicks scored", 5, len(shootout_goals))
check("England shootout goals", 3, int((shootout_goals.team == d.ENGLAND).sum()))
check("Sweden shootout goals", 2, int((shootout_goals.team == d.SWEDEN).sum()))
check("All period-5 shots are penalties", True, bool((shootout.shot_type == "Penalty").all()))

shootout[["minute", "second", "team", "player", "shot_outcome"]]

[OK      ] Shootout kicks: expected=14 actual=14 
[OK      ] Shootout kicks scored: expected=5 actual=5 
[OK      ] England shootout goals: expected=3 actual=3 
[OK      ] Sweden shootout goals: expected=2 actual=2 
[OK      ] All period-5 shots are penalties: expected=True actual=True 


,minute,second,team,player,shot_outcome
3858,120,2,England Women's,Alessia Russo,Goal
3860,120,46,Sweden Women's,Ingrid Filippa Angeldal,Saved
3862,121,32,England Women's,Lauren James,Saved
3864,122,21,Sweden Women's,Julia Zigiotti-Olme,Goal
3866,123,3,England Women's,Bethany Mead,Saved
3868,123,49,Sweden Women's,Magdalena Lilly Eriksson,Post
3870,124,30,England Women's,Alex Greenwood,Saved
3872,125,16,Sweden Women's,Gun Nathalie Björn,Goal
3874,126,20,England Women's,Chloe Kelly,Goal
3876,127,10,Sweden Women's,Jennifer Miley Falk,Off T


### 2.4 England substitutions

Known facts: Mead, Agyemang and Morgan came on after 70', Kelly at 77'.
"After 70'" is checked as `display_minute >= 70`; Kelly as `display_minute == 77`.

In [11]:
subs = d.get_substitutions(events)
eng_subs = subs[subs.team == d.ENGLAND]
eng_subs

,period,minute,second,display_minute,team,player_off,player_on,reason
1,2,69,9,70,England Women's,Ella Toone,Bethany Mead,Tactical
2,2,69,12,70,England Women's,Jessica Carter,Esme Beth Morgan,Tactical
3,2,69,29,70,England Women's,Georgia Stanway,Michelle Agyemang,Tactical
5,2,77,33,78,England Women's,Lauren Hemp,Chloe Kelly,Tactical
7,3,103,42,104,England Women's,Keira Walsh,Grace Clinton,Tactical
10,4,105,0,106,England Women's,Leah Williamson,Niamh Charles,Injury


In [12]:
def find_sub(surname: str):
    hit = eng_subs[eng_subs.player_on.str.contains(surname, na=False)]
    return None if hit.empty else hit.iloc[0]


for surname in ["Mead", "Agyemang", "Morgan"]:
    row = find_sub(surname)
    if row is None:
        check(f"{surname} on after 70'", True, None, "substitution not found")
        continue
    check(f"{surname} on after 70' (display minute >= 70)", True,
          bool(row.display_minute >= 70),
          f"display minute {row.display_minute}' {clock_str(row)}, replaced {row.player_off}")

row = find_sub("Kelly")
if row is None:
    check("Kelly on at 77'", "77'", None, "substitution not found")
else:
    check("Kelly on at 77'", "77'", f"{row.display_minute}'",
          f"{clock_str(row)}, replaced {row.player_off}")

[OK      ] Mead on after 70' (display minute >= 70): expected=True actual=True display minute 70' (clock 69:09, period 2), replaced Ella Toone
[OK      ] Agyemang on after 70' (display minute >= 70): expected=True actual=True display minute 70' (clock 69:29, period 2), replaced Georgia Stanway
[OK      ] Morgan on after 70' (display minute >= 70): expected=True actual=True display minute 70' (clock 69:12, period 2), replaced Jessica Carter
[MISMATCH] Kelly on at 77': expected="77'" actual="78'" (clock 77:33, period 2), replaced Lauren Hemp


### 2.5 Summary of the validation

In [13]:
summary = pd.DataFrame(results)
print(summary.status.value_counts().to_string())
summary

status
OK          17
MISMATCH     1


,check,expected,actual,status,detail
0,Asllani goal (Sweden Women's),2',2',OK,"(clock 01:44, period 1)"
1,Blackstenius goal (Sweden Women's),25',25',OK,"(clock 24:39, period 1)"
2,Bronze goal (England Women's),79',79',OK,"(clock 78:31, period 2)"
3,Agyemang goal (England Women's),81',81',OK,"(clock 80:13, period 2)"
4,Number of goals in periods 1-4,4,4,OK,
5,Sweden goals after extra time,2,2,OK,
6,England goals after extra time,2,2,OK,
7,Match sheet score (home-away),2-2,2-2,OK,
8,Extra time was played (periods 3 and 4 present),True,True,OK,
9,Shootout kicks,14,14,OK,


### Notes on mismatches

Any `MISMATCH` above is a genuine disagreement between StatsBomb's timestamps and the published
match facts; the data has not been altered. In particular, the StatsBomb clock places England's triple
substitution at 69:09–69:29 (the 70th minute) and Kelly's substitution at 77:33 (the 78th minute on a
broadcast-style count). Official match reports round substitution times differently, so a one-minute
difference is a timing-convention issue, not missing data. Later notebooks use the exact StatsBomb
timestamps (e.g. Kelly's substitution event) rather than hard-coded minutes.

## 3. Quick look at lineups and 360 coverage

In [14]:
names = d.player_display_names(lineups)
lineups.assign(display_name=lineups.player_name.map(names))[
    ["team", "jersey_number", "player_name", "display_name"]
].sort_values(["team", "jersey_number"])

,team,jersey_number,player_name,display_name
36,England Women's,1,Hannah Hampton,Hannah Hampton
26,England Women's,2,Lucy Bronze,Lucy Bronze
31,England Women's,3,Niamh Charles,Niamh Charles
25,England Women's,4,Keira Walsh,Keira Walsh
27,England Women's,5,Alex Greenwood,Alex Greenwood
32,England Women's,6,Leah Williamson,Leah Williamson
38,England Women's,7,Lauren James,Lauren James
23,England Women's,8,Georgia Stanway,Georgia Stanway
30,England Women's,9,Bethany Mead,Bethany Mead
39,England Women's,10,Ella Toone,Ella Toone


In [15]:
has_360 = events.id.isin(frames.id)
events.assign(has_360=has_360).groupby("period").has_360.mean().round(3).rename("share_with_360")

period
1    0.845
2    0.881
3    0.865
4    0.865
5    0.406
Name: share_with_360, dtype: float64